# Synthetic Pest Video — Batch Renderer
Run this notebook on **Google Colab (CPU runtime)**.
Open 5 copies simultaneously, change `SESSION_ID` to 0–4 in each one.

**Before running:** Upload to Google Drive:
- `pest_project/kitchens/`     ← `generator/kitchen_img/curated_img/` contents
- `pest_project/depth_cache/`  ← `depth_cache/` contents
- `pest_project/code.zip`      ← zipped `synthetic_video_gen/` repo

In [ ]:
# ── CHANGE THIS IN EACH TAB ──────────────────────────────────────────────────
SESSION_ID      = 0   # 0, 1, 2, 3, or 4
TOTAL_SESSIONS  = 5
VIDEOS_PER_KITCHEN = 20
SPLIT_SEED      = 42  # must be same across all sessions
# ─────────────────────────────────────────────────────────────────────────────

DRIVE_ROOT  = '/content/drive/MyDrive/pest_project'
IMAGE_DIR   = f'{DRIVE_ROOT}/kitchens'
DEPTH_CACHE = f'{DRIVE_ROOT}/depth_cache'
OUTPUT_DIR  = f'{DRIVE_ROOT}/renders'
CODE_ZIP    = f'{DRIVE_ROOT}/code.zip'
CODE_DIR    = '/content/synthetic_video_gen'

print(f'Session {SESSION_ID + 1} of {TOTAL_SESSIONS}')

In [ ]:
# ── 1. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── 2. Install dependencies ──────────────────────────────────────────────────
# Run once per session. Takes ~2 min.
!pip install -q mmengine timm scipy
!pip install -q 'opencv-python-headless' Pillow
print('Dependencies installed.')

In [ ]:
# ── 3. Unzip repo code ───────────────────────────────────────────────────────
import os, zipfile

if not os.path.exists(CODE_DIR):
    print('Unzipping code...')
    with zipfile.ZipFile(CODE_ZIP, 'r') as z:
        z.extractall('/content')
    # Handle if zip contains a top-level folder
    extracted = [d for d in os.listdir('/content') if 'synthetic_video_gen' in d.lower()]
    if extracted and extracted[0] != 'synthetic_video_gen':
        os.rename(f'/content/{extracted[0]}', CODE_DIR)
    print(f'Code extracted to {CODE_DIR}')
else:
    print('Code already present.')

# Verify key files exist
for f in ['scripts/render_batch_colab.py', 'generator/pipeline.py', 'generator/mmcv_stub']:
    status = '✓' if os.path.exists(f'{CODE_DIR}/{f}') else '✗ MISSING'
    print(f'  {status}  {f}')

In [ ]:
# ── 4. Verify Drive folders ──────────────────────────────────────────────────
import os

kitchens = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))]
caches   = [f for f in os.listdir(DEPTH_CACHE) if f.endswith('.npz')]

print(f'Kitchen images:  {len(kitchens)}')
print(f'Depth caches:    {len(caches)}')

if len(kitchens) == 0:
    raise RuntimeError(f'No images found in {IMAGE_DIR} — check Drive upload')
if len(caches) < len(kitchens) * 0.9:
    print(f'WARNING: only {len(caches)} caches for {len(kitchens)} images — some renders will run Metric3D (slow)')

In [ ]:
# ── 5. Preview kitchen split (optional, no renders) ──────────────────────────
!cd {CODE_DIR} && PYTHONPATH="generator/mmcv_stub:." python scripts/render_batch_colab.py \
    --image_dir   {IMAGE_DIR} \
    --depth_cache {DEPTH_CACHE} \
    --output_dir  {OUTPUT_DIR} \
    --n           {VIDEOS_PER_KITCHEN} \
    --seed        {SPLIT_SEED} \
    --session_id  {SESSION_ID} \
    --total_sessions {TOTAL_SESSIONS} \
    --list_splits

In [ ]:
# ── 6. RUN BATCH RENDERING ───────────────────────────────────────────────────
# This is the main cell. Runtime: ~1-2 hrs per session.
# Videos + annotations saved directly to Google Drive as they complete.
# Safe to re-run if interrupted — manifest tracks completed jobs.

!cd {CODE_DIR} && PYTHONPATH="generator/mmcv_stub:." python -u scripts/render_batch_colab.py \
    --image_dir      {IMAGE_DIR} \
    --depth_cache    {DEPTH_CACHE} \
    --output_dir     {OUTPUT_DIR} \
    --n              {VIDEOS_PER_KITCHEN} \
    --seed           {SPLIT_SEED} \
    --session_id     {SESSION_ID} \
    --total_sessions {TOTAL_SESSIONS}

In [ ]:
# ── 7. Check progress after rendering ────────────────────────────────────────
import json, os

manifest_path = f'{OUTPUT_DIR}/manifest.json'
if not os.path.exists(manifest_path):
    print('No manifest yet — rendering not started or output_dir wrong.')
else:
    m = json.load(open(manifest_path))
    kitchens_done = len(m.get('kitchens', {}))
    total_videos  = sum(len(v['job_ids']) for v in m['kitchens'].values())
    by_split = {}
    for info in m['kitchens'].values():
        s = info['split']
        by_split[s] = by_split.get(s, 0) + len(info['job_ids'])
    print(f'Kitchens rendered: {kitchens_done}')
    print(f'Total videos:      {total_videos}')
    print(f'By split:          {by_split}')

## After ALL 5 sessions finish

Run this **once** in any session to build the final YOLO dataset:

```python
!cd {CODE_DIR} && python scripts/build_dataset.py \
    --render_dir {OUTPUT_DIR} \
    --output_dir {DRIVE_ROOT}/pest_dataset \
    --every_n 2
```

Then train:

```python
!pip install -q ultralytics
!yolo train \
    data={DRIVE_ROOT}/pest_dataset/data.yaml \
    model=yolov8m.pt \
    epochs=100 imgsz=640 batch=16 device=0 \
    project={DRIVE_ROOT}/pest_runs name=yolov8m_pest_v1 \
    augment=True mosaic=1.0 degrees=10 fliplr=0.5 \
    hsv_h=0.015 hsv_s=0.5 hsv_v=0.4 translate=0.1 scale=0.3
```